In [18]:
import pandas as pd
import numpy as np
import yaml
import json
import logging
import time
import os

def run_mlops(input_file, config_file, output_file, log_file):
    start_time = time.time()

    # Logging setup
    logging.basicConfig(
        filename=log_file,
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s"
    )

    logging.info("Job started")

    try:
        # -----------------------
        # Validate config
        # -----------------------
        if not os.path.exists(config_file):
            raise FileNotFoundError("Config file not found")

        with open(config_file, "r") as f:
            config = yaml.safe_load(f)

        if not all(k in config for k in ["seed", "window", "version"]):
            raise ValueError("Config missing required fields")

        seed = config["seed"]
        window = config["window"]
        version = config["version"]

        np.random.seed(seed)
        logging.info(f"Config loaded: seed={seed}, window={window}, version={version}")

        # -----------------------
        # Validate dataset
        # -----------------------
        if not os.path.exists(input_file):
            raise FileNotFoundError("Input CSV file not found")

        df = pd.read_csv("data 1.csv")

        if df.empty:
            raise ValueError("Dataset is empty")

        if "close" not in df.columns:
            raise ValueError("Missing required column: close")

        logging.info(f"Rows loaded: {len(df)}")

        # -----------------------
        # Rolling mean
        # -----------------------
        df["rolling_mean"] = df["close"].rolling(window=window).mean()
        logging.info("Rolling mean calculated")

        # -----------------------
        # Signal generation
        # -----------------------
        df["signal"] = np.where(df["close"] > df["rolling_mean"], 1, 0)
        logging.info("Signal generation completed")

        # -----------------------
        # Metrics
        # -----------------------
        rows_processed = len(df)
        signal_rate = df["signal"].mean()
        latency_ms = int((time.time() - start_time) * 1000)

        metrics = {
            "version": version,
            "rows_processed": int(rows_processed),
            "metric": "signal_rate",
            "value": round(float(signal_rate), 4),
            "latency_ms": latency_ms,
            "seed": seed,
            "status": "success"
        }

        # Save metrics
        with open(output_file, "w") as f:
            json.dump(metrics, f, indent=2)

        logging.info(f"Metrics: {metrics}")
        logging.info("Job completed successfully")

        # Print metrics
        print(json.dumps(metrics, indent=2))

    except Exception as e:
        logging.exception("Job failed")

        metrics = {
            "version": config.get("version", "v1") if 'config' in locals() else "v1",
            "status": "error",
            "error_message": str(e)
        }

        with open(output_file, "w") as f:
            json.dump(metrics, f, indent=2)

        print(json.dumps(metrics, indent=2))
        # Do NOT call sys.exit() in Notebook


In [19]:

run_mlops(
    input_file="data.csv",
    config_file="config.yaml",
    output_file="metrics.json",
    log_file="run.log"
)

{
  "version": "v1",
  "rows_processed": 10000,
  "metric": "signal_rate",
  "value": 0.4989,
  "latency_ms": 81,
  "seed": 42,
  "status": "success"
}
